In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 7)

# helper to format large numbers
def fmt_b(val):
    """Format number to billions with 1 decimal."""
    if pd.isna(val):
        return "N/A"
    return f"{val / 1e9:.1f}B"


def fmt_pct(val):
    """Format decimal to percentage string."""
    if pd.isna(val):
        return "N/A"
    return f"{val * 100:.1f}%"

In [ ]:
# 6 large-caps across all 4 Nordic exchanges
# .ST = Stockholm, .CO = Copenhagen, .OL = Oslo, .HE = Helsinki
tickers = {
    "VOLV-B.ST": "Volvo",
    "SEB-A.ST": "SEB",
    "NOVO-B.CO": "Novo Nordisk",
    "EQNR.OL": "Equinor",
    "NOKIA.HE": "Nokia",
    "NESTE.HE": "Neste",
}

symbols = list(tickers.keys())
names = list(tickers.values())

In [ ]:
# Create Tickers object — this gives access to .info, financials, etc.
ytickers = yf.Tickers(" ".join(symbols))

# Collect .info for each company
info_data = {}
for sym, name in tickers.items():
    try:
        info_data[name] = ytickers.tickers[sym].info
    except Exception as e:
        print(f"Failed to fetch info for {name} ({sym}): {e}")

print(f"Fetched info for {len(info_data)} companies")

In [ ]:
overview_fields = {
    "Sector": "sector",
    "Industry": "industry",
    "Country": "country",
    "Market Cap": "marketCap",
    "Employees": "fullTimeEmployees",
    "Currency": "currency",
    "Trailing P/E": "trailingPE",
    "Forward P/E": "forwardPE",
    "P/B": "priceToBook",
    "Dividend Yield": "dividendYield",
    "Beta": "beta",
}

overview = pd.DataFrame(
    {
        name: {field: info.get(key) for field, key in overview_fields.items()}
        for name, info in info_data.items()
    }
).T

# Format for readability
overview["Market Cap"] = overview["Market Cap"].apply(
    lambda x: fmt_b(x) if pd.notna(x) else "N/A"
)
overview["Dividend Yield"] = overview["Dividend Yield"].apply(
    lambda x: fmt_pct(x) if pd.notna(x) else "N/A"
)
overview

In [ ]:
income_data = {}
for sym, name in tickers.items():
    try:
        stmt = ytickers.tickers[sym].income_stmt
        if stmt.empty:
            continue
        # Most recent annual period
        latest = stmt.iloc[:, 0]
        income_data[name] = {
            "Revenue": latest.get("Total Revenue"),
            "Gross Profit": latest.get("Gross Profit"),
            "EBITDA": latest.get("EBITDA"),
            "Operating Income": latest.get("Operating Income"),
            "Net Income": latest.get("Net Income"),
            "Period": stmt.columns[0].strftime("%Y-%m-%d") if hasattr(stmt.columns[0], "strftime") else str(stmt.columns[0]),
        }
    except Exception as e:
        print(f"Failed income stmt for {name}: {e}")

income_df = pd.DataFrame(income_data).T
income_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ["Revenue", "EBITDA", "Net Income"]
colors = ["steelblue", "darkorange", "seagreen"]

for ax, metric, color in zip(axes, metrics, colors):
    vals = income_df[metric].dropna().astype(float) / 1e9
    vals.plot(kind="barh", ax=ax, color=color)
    ax.set_title(metric)
    ax.set_xlabel("Billions (local currency)")

plt.suptitle("Income Statement — Latest Annual", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
margin_data = {}
for name, info in info_data.items():
    margin_data[name] = {
        "Gross Margin": info.get("grossMargins"),
        "Operating Margin": info.get("operatingMargins"),
        "Net Margin": info.get("profitMargins"),
    }

margin_df = pd.DataFrame(margin_data).T.astype(float) * 100  # to percent

fig, ax = plt.subplots(figsize=(12, 6))
margin_df.plot(kind="bar", ax=ax, color=["steelblue", "darkorange", "seagreen"])
ax.set_title("Profit Margins (%)")
ax.set_ylabel("%")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.legend(loc="upper right")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
balance_data = {}
for sym, name in tickers.items():
    try:
        bs = ytickers.tickers[sym].balance_sheet
        if bs.empty:
            continue
        latest = bs.iloc[:, 0]
        total_debt = latest.get("Total Debt", latest.get("Long Term Debt"))
        total_equity = latest.get("Total Equity Gross Minority Interest", latest.get("Stockholders Equity"))
        balance_data[name] = {
            "Total Assets": latest.get("Total Assets"),
            "Total Debt": total_debt,
            "Cash & Equivalents": latest.get("Cash And Cash Equivalents"),
            "Total Equity": total_equity,
            "Net Debt": (total_debt or 0) - (latest.get("Cash And Cash Equivalents") or 0),
            "Debt/Equity": (total_debt or 0) / total_equity if total_equity else None,
        }
    except Exception as e:
        print(f"Failed balance sheet for {name}: {e}")

balance_df = pd.DataFrame(balance_data).T

# Display in billions
display_cols = ["Total Assets", "Total Debt", "Cash & Equivalents", "Total Equity", "Net Debt"]
balance_display = balance_df.copy()
for col in display_cols:
    balance_display[col] = balance_df[col].apply(lambda x: fmt_b(x) if pd.notna(x) else "N/A")
balance_display["Debt/Equity"] = balance_df["Debt/Equity"].apply(
    lambda x: f"{x:.2f}" if pd.notna(x) else "N/A"
)
balance_display

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(balance_df))
width = 0.35

debt_vals = balance_df["Total Debt"].fillna(0).astype(float) / 1e9
cash_vals = balance_df["Cash & Equivalents"].fillna(0).astype(float) / 1e9

ax.bar(x - width / 2, debt_vals, width, label="Total Debt", color="indianred")
ax.bar(x + width / 2, cash_vals, width, label="Cash & Equivalents", color="mediumseagreen")
ax.set_xticks(x)
ax.set_xticklabels(balance_df.index, rotation=45, ha="right")
ax.set_ylabel("Billions (local currency)")
ax.set_title("Debt vs Cash Position")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
cf_data = {}
for sym, name in tickers.items():
    try:
        cf = ytickers.tickers[sym].cashflow
        if cf.empty:
            continue
        latest = cf.iloc[:, 0]
        operating_cf = latest.get("Operating Cash Flow")
        capex = latest.get("Capital Expenditure")
        fcf = latest.get("Free Cash Flow")
        if fcf is None and operating_cf is not None and capex is not None:
            fcf = operating_cf + capex  # capex is typically negative
        cf_data[name] = {
            "Operating CF": operating_cf,
            "Capital Expenditure": capex,
            "Free Cash Flow": fcf,
            "Dividends Paid": latest.get("Common Stock Dividend Paid"),
        }
    except Exception as e:
        print(f"Failed cash flow for {name}: {e}")

cf_df = pd.DataFrame(cf_data).T

fig, ax = plt.subplots(figsize=(12, 6))
cf_plot = cf_df[["Operating CF", "Free Cash Flow"]].astype(float) / 1e9
cf_plot.plot(kind="bar", ax=ax, color=["steelblue", "seagreen"])
ax.set_title("Cash Flow — Latest Annual")
ax.set_ylabel("Billions (local currency)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
valuation_fields = {
    "Trailing P/E": "trailingPE",
    "Forward P/E": "forwardPE",
    "P/B": "priceToBook",
    "P/S": "priceToSalesTrailing12Months",
    "EV/EBITDA": "enterpriseToEbitda",
    "EV/Revenue": "enterpriseToRevenue",
}

val_data = {}
for name, info in info_data.items():
    val_data[name] = {field: info.get(key) for field, key in valuation_fields.items()}

val_df = pd.DataFrame(val_data).T

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric in zip(axes, ["Trailing P/E", "EV/EBITDA", "P/B"]):
    vals = val_df[metric].dropna().astype(float)
    vals.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(metric)

plt.suptitle("Valuation Multiples", fontsize=14)
plt.tight_layout()
plt.show()

val_df

In [ ]:
div_data = {}
for name, info in info_data.items():
    div_data[name] = {
        "Dividend Yield (%)": round(info.get("dividendYield", 0) * 100, 2) if info.get("dividendYield") else "N/A",
        "Payout Ratio (%)": round(info.get("payoutRatio", 0) * 100, 2) if info.get("payoutRatio") else "N/A",
        "5Y Avg Yield (%)": round(info.get("fiveYearAvgDividendYield", 0), 2) if info.get("fiveYearAvgDividendYield") else "N/A",
        "Ex-Dividend Date": info.get("exDividendDate"),
    }

div_df = pd.DataFrame(div_data).T
div_df

In [ ]:
price_data = yf.download(symbols, period="5y", auto_adjust=True)
close = price_data["Close"].dropna(how="all").rename(columns=tickers)
normalized = close / close.iloc[0] * 100

fig, ax = plt.subplots()
normalized.plot(ax=ax, alpha=0.8)
ax.set_title("5-Year Price Performance (base = 100)")
ax.set_ylabel("Indexed Price")
ax.axhline(100, color="black", linestyle="--", linewidth=0.8)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

---
## Deep Dive — Single Stock
Change `target` below to any ticker from the dict above to get
full quarterly financials for that company.

In [ ]:
target_sym = "NOVO-B.CO"
target_name = tickers[target_sym]
t = yf.Ticker(target_sym)

print(f"Deep dive: {target_name} ({target_sym})")
print(f"Sector: {t.info.get('sector')} | Industry: {t.info.get('industry')}")
print(f"Market Cap: {fmt_b(t.info.get('marketCap'))}")
print(f"Description: {t.info.get('longBusinessSummary', 'N/A')[:300]}...")

In [ ]:
q_income = t.quarterly_income_stmt
print(f"\n=== Quarterly Income Statement — {target_name} ===")
print(f"Periods: {[c.strftime('%Y-%m-%d') for c in q_income.columns]}\n")

key_rows = [
    "Total Revenue",
    "Cost Of Revenue",
    "Gross Profit",
    "Operating Expense",
    "Operating Income",
    "EBITDA",
    "Net Income",
]
q_income_filtered = q_income.loc[q_income.index.isin(key_rows)]
(q_income_filtered / 1e9).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
rev = q_income.loc["Total Revenue"].sort_index().astype(float) / 1e9
ni = q_income.loc["Net Income"].sort_index().astype(float) / 1e9

ax.bar(rev.index.strftime("%Y-Q%q") if hasattr(rev.index, "strftime") else range(len(rev)),
       rev.values, color="steelblue", alpha=0.7, label="Revenue")
ax.plot(range(len(ni)), ni.values, color="seagreen", marker="o", linewidth=2, label="Net Income")
ax.set_xticks(range(len(rev)))
ax.set_xticklabels([d.strftime("%Y-Q%q") if hasattr(d, "strftime") else str(d) for d in rev.index], rotation=45, ha="right")
ax.set_ylabel("Billions")
ax.set_title(f"{target_name} — Quarterly Revenue & Net Income")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
q_balance = t.quarterly_balance_sheet
print(f"\n=== Quarterly Balance Sheet — {target_name} ===\n")

bs_rows = [
    "Total Assets",
    "Total Debt",
    "Long Term Debt",
    "Cash And Cash Equivalents",
    "Total Equity Gross Minority Interest",
    "Stockholders Equity",
]
q_balance_filtered = q_balance.loc[q_balance.index.isin(bs_rows)]
(q_balance_filtered / 1e9).round(2)

In [ ]:
q_cf = t.quarterly_cashflow
print(f"\n=== Quarterly Cash Flow — {target_name} ===\n")

cf_rows = [
    "Operating Cash Flow",
    "Capital Expenditure",
    "Free Cash Flow",
    "Common Stock Dividend Paid",
    "Repurchase Of Capital Stock",
]
q_cf_filtered = q_cf.loc[q_cf.index.isin(cf_rows)]
(q_cf_filtered / 1e9).round(2)